In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce.silver")

In [0]:
orders = spark.read.table("ecommerce.bronze.orders")
customers = spark.read.table("ecommerce.bronze.customers")
items = spark.read.table("ecommerce.bronze.order_items")
products = spark.read.table("ecommerce.bronze.products")
payments = spark.read.table("ecommerce.bronze.order_payments")
reviews = spark.read.table("ecommerce.bronze.order_reviews")
sellers = spark.read.table("ecommerce.bronze.sellers")
geolocation = spark.read.table("ecommerce.bronze.geolocation")

In [0]:
def standardize_col_name(col_name):
    return col_name.strip().replace(' ', '_').lower()

def rename_columns(df):
    return df.toDF(*[standardize_col_name(c) for c in df.columns])

orders = rename_columns(orders)
customers = rename_columns(customers)
items = rename_columns(items)
products = rename_columns(products)
payments = rename_columns(payments)
reviews = rename_columns(reviews)
sellers = rename_columns(sellers)
geolocation = rename_columns(geolocation)

In [0]:
products = products.withColumnRenamed(
    "product_name_lenght", "product_name_length"
)
products = products.withColumnRenamed(
    "product_description_lenght", "product_description_length"
)

In [0]:
customers = customers.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("string"))
sellers = sellers.withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("string"))
geolocation = geolocation.withColumn("geolocation_zip_code_prefix", col("geolocation_zip_code_prefix").cast("string"))

In [0]:
geolocation = geolocation.withColumn(
    "geolocation_city",
    translate(col("geolocation_city"), "ãáâàäéêèëíîìïóôòöõúûùüçÃÁÂÀÄÉÊÈËÍÎÌÏÓÔÒÖÕÚÛÙÜÇ", "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC")
)

In [0]:
items = items.dropDuplicates(["order_id", "order_item_id"])
reviews = reviews.dropDuplicates(["review_id"])

# Remove corrupted review rows where IDs have incorrect lengths (valid UUIDs are 32 chars)
reviews = reviews.filter(
    (length(col("review_id")) == 32) & (length(col("order_id")) == 32)
)

In [0]:
customers = customers.fillna({
    "customer_city": "unknown",
    "customer_state": "unknown"
})

products = products.fillna({
    "product_category_name": "unknown"
})


In [0]:
display(orders)

In [0]:
products = products.withColumn(
    "missing_category_flag",
    when(col("product_category_name") == "unknown", 1).otherwise(0)
)
display(products)

In [0]:
# Orders - convert timestamp strings to dates
orders = orders \
    .withColumn("order_purchase_timestamp", to_date(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at", to_date(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date", to_date(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date", to_date(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_date(col("order_estimated_delivery_date")))


# Reviews - convert timestamp strings to dates
reviews = reviews \
    .withColumn("review_creation_date", try_to_date(col("review_creation_date"))) \
    .withColumn("review_answer_timestamp", try_to_date(col("review_answer_timestamp")))


# Items - convert timestamp string to date
items = items \
    .withColumn("shipping_limit_date", to_date(col("shipping_limit_date")))

In [0]:
display(reviews)

In [0]:
items = items \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

payments = payments \
    .withColumn("payment_value", col("payment_value").cast("double"))

In [0]:
payments_agg = payments.groupBy("order_id") \
    .agg(sum("payment_value").alias("total_payment"))

In [0]:
orders = orders.withColumn(
    "is_late_delivery",
    col("order_delivered_customer_date") > col("order_estimated_delivery_date")
)

orders = orders.withColumn(
    "invalid_delivery",
    col("order_delivered_customer_date") < col("order_purchase_timestamp")
)

orders = orders.withColumn(
    "delivery_missing",
    col("order_delivered_customer_date").isNull()
)


In [0]:
orders = orders \
    .withColumn("order_year", year("order_purchase_timestamp")) \
    .withColumn("order_month", month("order_purchase_timestamp"))

In [0]:
# Join orders with customers (inner join ensures referential integrity)
orders = orders.join(customers, "customer_id", "inner")

# Join items with products and sellers (inner join ensures referential integrity)
items = items.join(products, "product_id", "inner")
items = items.join(sellers, "seller_id", "inner")

In [0]:
orders.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.silver.orders")
customers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.silver.customers")
items.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.silver.order_items")
products.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.silver.products")
payments_agg.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.silver.payments")
reviews.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.silver.reviews")
sellers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.silver.sellers")
geolocation.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.silver.geolocation")